# A Quick Introduction to UNIX

**Advanced Computational Physics** &middot; Master in Physics &middot; USC

*J. A. Hernando Morata*

This notebook is an *interactive* guide: every command below can be run and modified.

## What you will learn

- What a shell is, and why you need one
- Moving around the file system
- Creating, copying, moving and removing files
- Looking inside files
- Wildcards, redirection and pipes
- Searching: `grep` and `find`
- Permissions, environment variables, processes
- Working on a remote machine
- Configuring your shell, and writing a first script

## 1. Why the shell?

In physics you will meet machines you can only reach through a terminal:
computing clusters, remote servers, data acquisition systems.

- It is the **only** interface on most compute clusters.
- It is **scriptable**: anything you do by hand can be automated and repeated.
- It **composes**: small tools chain together to do large jobs.

A *shell* is the program that reads what you type and asks the operating system to do it.
The most common one is **bash**.

## 2. Running shell commands from Jupyter

In a notebook, a line starting with `!` is sent to the shell:

```python
!ls
```

Each `!` line runs in **its own subshell**, which is why `!cd somewhere` has no lasting effect.
To change the notebook's own directory use the magic `%cd`.

In a real terminal you would type the command **without** the `!`.

## 3. A sandbox to play in

Everything in this notebook happens inside a folder called `unix_sandbox`,
so that nothing else on your computer is touched. Running this cell again
starts from a clean state.

In [ ]:
import os, shutil

BASE = globals().get("BASE", os.getcwd())      # remember where we started
os.chdir(BASE)
shutil.rmtree("unix_sandbox", ignore_errors=True)
os.makedirs("unix_sandbox")
os.chdir("unix_sandbox")
print("Working inside:", os.path.basename(os.getcwd()) + "/")

## 4. Where am I?

- `pwd` &mdash; *print working directory*
- `ls` &mdash; list the contents of a directory

In [ ]:
!pwd

In [ ]:
!ls -l

### Useful options for `ls`

| Option | Meaning |
|---|---|
| `-l` | long format: permissions, owner, size, date |
| `-h` | human-readable sizes (`4.0K`, `2.3M`) |
| `-a` | show hidden files (those whose name starts with `.`) |
| `-t` | sort by modification time, newest first |
| `-R` | descend into subdirectories |

Options can be combined: `ls -lha`.

In [ ]:
!ls -lha

## 5. Paths

The file system is a **tree**. A path says how to reach a file in that tree.

| Path | Meaning |
|---|---|
| `/home/ana/data.txt` | **absolute**: starts at the root `/` |
| `data/run1.txt` | **relative**: starts at the current directory |
| `.` | the current directory |
| `..` | the parent directory |
| `~` | your home directory |

An absolute path always means the same file. A relative path depends on where you are.

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — Where does it point?</b></p>

You are in `/home/ana/fca/data`. **Decide** which absolute path each of these means:
`../notes/a.txt`, `./b.txt`, `b.txt`, `../../c.txt`, `~/d.txt`.

</div>

<details>
<summary><b>Solution</b></summary>

`/home/ana/fca/notes/a.txt`, `/home/ana/fca/data/b.txt` (twice: `./` is optional),
`/home/ana/c.txt`, and `/home/ana/d.txt` (assuming `ana`'s home is `/home/ana`).

</details>

In [ ]:
!echo "Home directory: $HOME"
!ls ..

### Moving around: `cd`

```bash
cd data          # go into ./data
cd ..            # go up one level
cd ~             # go home
cd -             # go back to the previous directory
cd               # with no argument, also goes home
```

Remember: inside Jupyter use `%cd`, because `!cd` only affects its own subshell.

## 6. Getting help

| Command | What it does |
|---|---|
| `man ls` | the manual page for `ls` (press `q` to quit) |
| `ls --help` | short help (GNU/Linux only &mdash; see *POSIX*, later) |
| `apropos copy` | search the manuals by keyword |
| `which python3` | where does this command live? |
| `type cd` | is it a program, a builtin, an alias? |

`man` is a *pager*: it takes over the terminal, so it does not work well inside a notebook.

In [ ]:
!man ls | col -b | head -12

## 7. Creating files and directories

- `mkdir name` &mdash; make a directory
- `mkdir -p a/b/c` &mdash; make the whole chain, and do not complain if it exists
- `touch file` &mdash; create an empty file (or update its timestamp)

In [ ]:
!mkdir -p data/raw
!touch data/raw/run1.txt data/raw/run2.txt
!ls -R data

## 8. Removing files and directories

- `rm file` &mdash; remove a file
- `rmdir dir` &mdash; remove an **empty** directory
- `rm -r dir` &mdash; remove a directory and everything in it

> ⚠️ **There is no recycle bin.** `rm` is immediate and irreversible.
> Be especially careful with `rm -rf`, and never run it on a path you have not read twice.
> `rm -i` asks for confirmation on each file.

In [ ]:
!touch scratch.txt
!ls
!rm scratch.txt
!ls

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop (review) — One variable</b></p>

A script contains this line, to clean a temporary folder:

```bash
rm -rf $TMPDIR/
```

What happens if the variable `TMPDIR` was never defined?

</div>

<details>
<summary><b>Solution</b></summary>

An undefined variable expands to nothing, and the line becomes `rm -rf /`: an attempt to
delete **everything**. This has happened, in real software. Defences: check the variable
first, write `${TMPDIR:?not set}` (the shell stops with an error if it is empty), and
never run `rm -rf` on a path built from variables you have not printed.

</details>

## 9. Copying and moving

- `cp source target` &mdash; copy
- `cp -r dir target` &mdash; copy a whole directory
- `mv source target` &mdash; move **or** rename (it is the same operation)

Both will **overwrite the target without asking**; `-i` makes them ask.

In [ ]:
!echo "Hello World"    > data/raw/run1.txt
!echo "This is a test"  > data/raw/run2.txt

!cp data/raw/run1.txt data/raw/run1_copy.txt
!mv data/raw/run2.txt data/raw/run2_renamed.txt
!ls data/raw

## 10. Looking inside a file

| Command | What it shows |
|---|---|
| `cat file` | the whole file |
| `head -5 file` | the first 5 lines |
| `tail -5 file` | the last 5 lines |
| `tail -f file` | the end of the file, updating live (useful for logs) |
| `less file` | page through it (`q` to quit, `/` to search) |
| `wc -l file` | how many lines |

For a large file, never use `cat`: use `less`, `head` or `tail`.

In [ ]:
!cat data/raw/run1.txt

In [ ]:
# build a slightly larger file to play with
!seq 1 100 > numbers.txt
!head -3 numbers.txt
!echo "..."
!tail -3 numbers.txt
!wc -l numbers.txt

## 11. Wildcards

The shell expands these **before** the command runs:

| Pattern | Matches |
|---|---|
| `*` | any number of characters (including none) |
| `?` | exactly one character |
| `[0-9]` | one character from the set |
| `{a,b}` | either alternative |

So `ls *.txt` lists every file ending in `.txt`.

In [ ]:
!ls data/raw/*.txt
!echo "---"
!ls data/raw/run?.txt

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — Which files?</b></p>

A folder contains `run1.txt`, `run2.txt`, `run10.txt` and `notes.md`. **Decide** what
each command lists: `ls run?.txt`, `ls run*.txt`, `ls *1*`.

</div>

In [ ]:
!mkdir -p wild && cd wild && touch run1.txt run2.txt run10.txt notes.md && echo "run?.txt:" && ls run?.txt && echo "run*.txt:" && ls run*.txt && echo "*1*:" && ls *1*

<details>
<summary><b>Solution</b></summary>

`run?.txt` matches exactly one character after `run`: `run1.txt run2.txt`.
`run*.txt` matches any number: the three runs. `*1*` matches any name containing a `1`:
`run1.txt run10.txt`. And `ls` sorts alphabetically: `run10.txt` comes before `run2.txt`.
Name your files `run01`, `run02`… and the order will be right.

</details>

## 12. Redirection

Every command has an input and two outputs. You can redirect them.

| Syntax | Meaning |
|---|---|
| `cmd > file` | send output to `file`, **overwriting** it |
| `cmd >> file` | **append** the output to `file` |
| `cmd < file` | read input from `file` |
| `cmd 2> file` | redirect error messages |
| `cmd > file 2>&1` | redirect output *and* errors to the same file |

In [ ]:
!echo "first line"  > log.txt
!cat log.txt
!echo "----"
!echo "second line" >> log.txt
!cat log.txt

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — Sort in place?</b></p>

We want to sort `log.txt` and keep the result in the same file. **Before running**, what
does this leave in `log.txt`?

```bash
sort log.txt > log.txt
```

</div>

In [ ]:
!sort log.txt > log.txt
!wc -l log.txt

<details>
<summary><b>Solution</b></summary>

An empty file. The shell sets up the redirection **before** running the command:
`> log.txt` empties the file first, and then `sort` reads an empty file. Write to a
different file and rename it (`sort log.txt > tmp && mv tmp log.txt`), or use
`sort -o log.txt log.txt`, which is designed for this.

</details>

## 13. Pipes

A pipe `|` sends the output of one command straight into the next.
This is the idea that makes the shell powerful: small tools, combined.

```bash
ls -l | wc -l                 # how many entries in this directory
cat data.txt | sort | uniq    # sorted, without repetitions
```

In [ ]:
!seq 1 20 | tail -5 | wc -l

In [ ]:
# a small pipeline: the 3 most frequent words
!printf "alpha\nbeta\nalpha\ngamma\nbeta\nalpha\n" > words.txt
!sort words.txt | uniq -c | sort -rn | head -3

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop — Pipes on a data file</b></p>

The next cell writes `events.csv`, one line per event: `run,energy`. Using only shell
commands and pipes:

1. How many events are there?
2. How many events belong to run 2?
3. Which are the three highest energies?
4. Which runs are in the file, without repetitions?

*Useful:* `wc -l`, `grep -c`, `cut -d, -f2`, `sort -g -r`, `sort -u`, `head`.

</div>

In [ ]:
import random

rng = random.Random(1)
with open("events.csv", "w") as f:
    for _ in range(500):
        f.write(f"{rng.choice([1, 2, 3])},{rng.expovariate(1 / 50.):.2f}\n")
!head -3 events.csv

In [ ]:
# your commands here

<details>
<summary><b>Solution</b></summary>

```bash
wc -l events.csv                                   # 1
grep -c "^2," events.csv                           # 2
cut -d, -f2 events.csv | sort -g -r | head -3      # 3
cut -d, -f1 events.csv | sort -u                   # 4
```

Note `"^2,"`: a run number at the beginning of the line (`^`), followed by the comma.
`grep -c 2` would also count every energy containing a `2`. And `sort -g` sorts numbers
as numbers; plain `sort` would put `100.5` before `20.3`.

</details>

## 14. Searching *inside* files: `grep`

`grep PATTERN file` prints the lines that match.

| Option | Meaning |
|---|---|
| `-i` | ignore case |
| `-n` | show line numbers |
| `-r` | search recursively in a directory |
| `-v` | invert: show lines that do **not** match |
| `-c` | count matches instead of printing them |

In [ ]:
!grep -n "alpha" words.txt
!echo "---"
!grep -c "alpha" words.txt

## 15. Searching *for* files: `find`

```bash
find . -name "*.txt"          # by name, from here down
find . -type d                # only directories
find . -name "*.log" -delete  # careful!
find . -mtime -1              # modified in the last day
```

In [ ]:
!find . -name "*.txt" 

## 16. Permissions

`ls -l` shows something like `-rw-r--r--`:

```
 -    rw-      r--      r--
type  owner    group    others
```

- `r` read &middot; `w` write &middot; `x` execute (for a directory: *enter*)
- `chmod u+x script.sh` &mdash; give the **u**ser permission to e**x**ecute
- `chmod 644 file` &mdash; the same in octal: owner `rw-`, everyone else `r--`

You will need this the first time a script of yours refuses to run.

In [ ]:
!touch demo.sh
!ls -l demo.sh
!chmod u+x demo.sh
!ls -l demo.sh

<div class="admonition important">
<p class="admonition-title"><b>[=] Commit — In octal</b></p>

**Decide** what `chmod 750 analysis.sh` allows to the owner, to the group and to the
others. (`r = 4`, `w = 2`, `x = 1`.)

</div>

<details>
<summary><b>Solution</b></summary>

`7 = 4 + 2 + 1`: the owner can read, write and execute; `5 = 4 + 1`: the group can read
and execute; `0`: the others cannot do anything. `ls -l` shows `-rwxr-x---`.

</details>

## 17. Environment variables

Variables the shell passes to every program it runs.

| Variable | Meaning |
|---|---|
| `$HOME` | your home directory |
| `$PATH` | where the shell looks for commands |
| `$SHELL` | which shell you are using |
| `$USER` | your login name |

```bash
export MY_DATA=/scratch/run42     # define one
echo $MY_DATA                     # use it
```

In [ ]:
!echo "shell: $SHELL"
!echo "user: $USER"
!echo "home: $HOME"
!echo "path: $PATH"
!echo "----"
!echo "First three entries of PATH:"
!echo $PATH | tr ':' '\n' | head -3

In [ ]:
!which python3
!which ls

<div class="admonition note">
<p class="admonition-title"><b>[!] Discuss — Why ./script.sh?</b></p>

To run a script in the current directory you must write `./script.sh`, not `script.sh`.
It would be more convenient to add `.` to `$PATH`. Why is that a bad idea?

</div>

<details>
<summary><b>Solution</b></summary>

With `.` in `$PATH`, typing `ls` in a directory that contains a file called `ls` would
run **that** file, which anybody could have put there. It also makes the result of a
command depend on where you are. Being explicit, `./`, removes both problems.

</details>

## 18. Processes

| Command | What it does |
|---|---|
| `ps` | list your processes |
| `top` / `htop` | live view of what is running |
| `kill 1234` | ask process 1234 to stop |
| `kill -9 1234` | force it to stop |
| `cmd &` | run `cmd` in the background |
| `Ctrl-C` | interrupt the running command |
| `Ctrl-Z` then `bg` | suspend it, then resume in the background |

In [ ]:
!ps -o pid,comm | head -5

## 19. Working on a remote machine

This is how you will use a computing cluster.

```bash
ssh user@server.usc.es              # open a remote shell
scp data.txt user@server:/scratch/  # copy a file there
scp user@server:/scratch/out.dat .  # and back
rsync -av data/ user@server:/data/  # copy only what changed
```

None of these can be demonstrated inside the notebook: they need a real terminal.

### Keeping a job alive: `screen` and `tmux`

If your connection drops, everything you were running dies with it &mdash; unless
you started it inside a `screen` or `tmux` session.

```bash
screen -S mysession    # start a named session
# ... launch your long job ...
Ctrl-a  d              # detach: the job keeps running
screen -r mysession    # reconnect later, from anywhere
screen -ls             # list your sessions
```

`tmux` does the same with a nicer interface; the detach key is `Ctrl-b d`.

## 20. The common UNIX shells

| Shell | Strengths | Weaknesses | Best for |
|---|---|---|---|
| **sh** (Bourne) | Minimal, available everywhere | Very limited interactively | Legacy scripts |
| **bash** (Bourne Again) | Standard, POSIX-friendly, scripting workhorse | Lacks some modern conveniences | Teaching, scripting, servers |
| **zsh** (Z shell) | Smart completion, plugins, themes | Not always preinstalled | Daily work, power users |
| **ksh** (Korn) | POSIX compliant, stable | Less common nowadays | Legacy UNIX systems |
| **csh / tcsh** | C-like syntax, history | Poor for scripting, outdated | Historical interest |
| **fish** | Friendly, autosuggestions | Not POSIX compatible | Personal productivity |

Since macOS Catalina the default is **zsh**; most Linux distributions still default to **bash**.

**POSIX** (*Portable Operating System Interface*) is the standard that defines the
shell syntax and the utilities used throughout this notebook: stay within it and
your scripts run everywhere.

### What POSIX guarantees &mdash; and what it does not

POSIX (IEEE 1003) is the agreement that ended the divergence between the UNIX
variants of the 1980s. It fixes three things:

- the **C system calls** (`open`, `read`, `fork`, `kill`) &mdash; why `ps` and `Ctrl-C` behave alike everywhere;
- the **shell language**: `|`, `>`, `>>`, `2>&1`, `$VAR`, `for`, `if`;
- a **core set of utilities**: `ls`, `cp`, `grep`, `find`, `sed`, with a minimum set of options.

But the standard fixes the *minimum*, not the maximum, and every system adds its own
extensions. This is where macOS and Linux part company:

| Works on Linux | Fails on macOS | Why |
|---|---|---|
| `ls --help` | no such option | long options are a GNU extension |
| `sed -i 's/a/b/' f` | needs an argument: `sed -i '' ...` | BSD `sed` differs from GNU `sed` |

> **Rule of thumb.** Start scripts with `#!/bin/bash`, prefer short one-letter options,
> and distrust long ones such as `--color`. Everything taught in this notebook is
> inside that safe core.

***[+] Lookout.*** [explainshell.com](https://explainshell.com) takes any command line and explains each
piece; [tldr pages](https://tldr.sh) give short, practical examples instead of the full
`man` page.

In [ ]:
!echo $SHELL

## 21. Configuring your shell

Your shell reads a configuration file every time it starts:

| Shell | File |
|---|---|
| bash | `~/.bashrc` (and `~/.bash_profile` on login) |
| zsh | `~/.zshrc` |

Typical contents:

```bash
alias ll='ls -lh'                        # a shortcut
alias ..='cd ..'
export MY_PROJECT=$HOME/projects/myapp   # a variable
export PATH=$HOME/bin:$PATH              # add your own programs
```

After editing it, run `source ~/.bashrc` to apply the changes to the current shell.

> ⚠️ **Do not let a notebook write into your `~/.bashrc`.**
> A cell that appends to it adds the same lines again every time you run it,
> and a broken `.bashrc` can leave you with a shell that will not start.
> Edit it by hand, with an editor, and keep a copy.

The cell below only **shows** the end of your configuration file; it changes nothing.

In [ ]:
import os

for name in ("~/.zshrc", "~/.bashrc"):
    path = os.path.expanduser(name)
    if os.path.exists(path):
        n = len(open(path).read().splitlines())
        print(f"You have a {name}, {n} lines long. Open it with an editor to see it.")
        break
else:
    print("Neither ~/.zshrc nor ~/.bashrc exists on this machine.")

## 22. Your first shell script

A script is just a file with commands in it.

1. The first line, the **shebang**, says which interpreter to use.
2. Make it executable with `chmod +x`.
3. Run it with `./script.sh`.

Note `./`: the shell does **not** look in the current directory unless you say so.

In [ ]:
%%writefile hello.sh
#!/bin/bash
# A first script
echo "Hello from $(basename $0)"
echo "Today is $(date +%Y-%m-%d)"
echo "There are $(ls | wc -l) entries in $(basename $(pwd))/"

In [ ]:
!chmod +x hello.sh
!./hello.sh

## 23. Practice

Do them here, inside the sandbox, or in a real terminal.

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop — Warm up</b></p>

1. Create a directory `FCA`, move into it, create a file `README.md` containing the line
   `# Advanced Computational Physics`, and show its contents.
2. Create ten files `run01.txt` … `run10.txt` with a single command.
   *Hint:* `touch run{01..10}.txt`.
3. Count how many lines in `/etc/passwd` contain the string `bin`, using a pipe.

</div>

In [ ]:
# your commands here

<details>
<summary><b>Solution</b></summary>

```bash
mkdir FCA && cd FCA && echo "# Advanced Computational Physics" > README.md && cat README.md
touch run{01..10}.txt
cat /etc/passwd | grep bin | wc -l        # or simply: grep -c bin /etc/passwd
```

In the notebook, the first line must be a single `!` line, because each `!` runs in its
own subshell and the `cd` would be lost.

</details>

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop (review) — Four mistakes</b></p>

What is wrong in each case?

```bash
cd data/                 # and then the next command says: No such file or directory
rm *.txt file.txt
./script.sh              # Permission denied
echo "result" > out.txt  # run twice, and only one line is there
```

</div>

<details>
<summary><b>Solution</b></summary>

1. The command after `cd data/` runs **inside** `data/`; relative paths are now relative
   to it (or `data/` does not exist where you are: check with `pwd`).
2. `*.txt` already includes `file.txt`: `rm` will complain that it does not exist, after
   deleting it. Worse: a stray space, `rm * .txt`, deletes everything.
3. The script has no execute permission: `chmod u+x script.sh`.
4. `>` overwrites; `>>` appends.

</details>

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop — A script with an argument</b></p>

Write a script `count.sh` that takes a directory as its argument and prints how many
`.txt` files it contains. Inside a script, `$1` is the first argument. What should it do
if no argument is given?

</div>

In [ ]:
# your code here

<details>
<summary><b>Solution</b></summary>

```bash
#!/bin/bash
# count.sh: number of .txt files in a directory
if [ -z "$1" ]; then
    echo "usage: $0 directory" >&2
    exit 1
fi
ls "$1"/*.txt 2> /dev/null | wc -l
```

Then `chmod +x count.sh` and `./count.sh data/raw`. The quotes in `"$1"` protect names
with spaces; `2> /dev/null` hides the error when there are no `.txt` files.

</details>

<div class="admonition tip">
<p class="admonition-title"><b>[>] Workshop (in pairs) — Explain it to me</b></p>

In pairs. Each one writes on paper a one-line pipeline of at least three commands, for
example `find . -name "*.txt" | xargs grep -l alpha | wc -l`. Swap: explain your
partner's line **without running it**, step by step. Then run it and check.

</div>

## 24. Cleaning up

Everything we made lives in `unix_sandbox`. This removes it.

> Run this **only when you are done**. Afterwards the sandbox is gone, and the
> cells above will fail until you re-run the setup cell that created it.

In [ ]:
import os, shutil

BASE = globals().get("BASE", os.getcwd())      # also works after a kernel restart
os.chdir(BASE)
shutil.rmtree("unix_sandbox", ignore_errors=True)
print("Sandbox removed. Back in:", os.path.basename(os.getcwd()) + "/")

## 25. Summary

* The shell is the interface of clusters and servers: small tools, combined with pipes.
* Paths: absolute from `/`, relative from where you are.
* `>` overwrites, `>>` appends; the shell expands wildcards and variables **before** the
  command runs.
* There is no recycle bin.

<details>
<summary><b>[i] Cheat sheet — UNIX</b></summary>

| | |
|:--|:--|
| `pwd` `ls` `cd` | where am I, what is here, go there |
| `mkdir` `touch` `rm` `rmdir` | create and destroy |
| `cp` `mv` | copy, move/rename |
| `cat` `head` `tail` `less` `wc` | look inside |
| `*` `?` `[ ]` | wildcards |
| `>` `>>` `<` `\|` | redirection and pipes |
| `grep` `find` | search inside files / for files |
| `chmod` `ls -l` | permissions |
| `echo $VAR` `export` `which` | environment |
| `ps` `kill` `Ctrl-C` `&` | processes |
| `ssh` `scp` `screen` | remote work |
| `man` `--help` | when in doubt |

</details>

### Check yourself

1. What is the difference between `/data/run1.txt` and `data/run1.txt`?
2. Why does `!cd data` in a notebook not change the directory?
3. What does `cmd > file 2>&1` do?
4. Why is `sort f > f` a bug?
5. What does `chmod 644 file` allow?

<details>
<summary><b>Solutions</b></summary>

1. The first is absolute (from the root); the second is relative to the current
   directory.
2. Each `!` line runs in its own subshell, which ends with the line. Use `%cd`.
3. Sends the output and the error messages of `cmd` to `file`.
4. The shell empties `f` before `sort` reads it.
5. Owner: read and write; group and others: read only.

</details>